# Estimating effects with circles

In [1]:
import os
import sys
sys.path.insert(1, '../')

import warnings
from ps_matching import PSMatching
from effect_estimation import EffectEstimation
from ps_features_builder import PSFeaturesBuilder

warnings.filterwarnings("ignore")

PATH_DATA = '../../data/'
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet')
}

In [2]:
radii = [ # radio, número de círculos alrededor creados para tener buen match
    37.5, # 25
    50, # 15
    100, # 20
    150, # 15
    200, # 20
    250, # 25
]
radius = radii[-1]
print(f"Processing radius: {radius}")

Processing radius: 250


In [3]:
ps_builder = PSFeaturesBuilder(
    paths=PATHS,
    grid_type="circular",
    circle_radius=50,
    n_circles=15
)
ps_builder.build()

In [6]:
ps_builder.ps_features

,grid_id,has_camera,n_vialidades,max_carriles,both_directions,via_primaria,via_acc_cont,max_nivel,road_length_m,mean_afluencia_mensual,std_afluencia_mensual,distance_to_station,mean_total,std_total,mean_min,std_min,mean_pic,std_pic,mean_fcs,std_fcs
0,0,1,1,3.0,0,0,1,0.0,495.14,17.144257,2.139368,200.866760,0.054795,0.247394,0.041096,0.220821,0.013699,0.116503,0.000000,0.000000
1,1,1,3,6.0,0,1,1,0.0,403.88,22.950475,3.281180,636.481693,0.835616,1.628420,0.543379,1.487467,0.287671,0.869664,0.004566,0.067574
2,2,1,1,4.0,0,1,0,0.0,196.86,112.081967,15.246665,584.344738,0.305936,0.724734,0.150685,0.524743,0.155251,0.544874,0.000000,0.000000
3,3,1,2,4.0,1,1,0,0.0,235.18,40.207765,6.022594,338.683674,0.383562,0.783373,0.168950,0.519124,0.205479,0.641649,0.009132,0.095344
4,4,1,1,3.0,0,0,1,2.0,687.37,111.886988,17.401815,1446.490506,0.041096,0.198967,0.036530,0.188034,0.004566,0.067574,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25947,25947,0,1,5.0,1,1,0,0.0,91.40,80.342039,11.148066,1084.073290,0.050228,0.238953,0.018265,0.134214,0.031963,0.200644,0.000000,0.000000
25948,25948,0,1,5.0,1,1,0,0.0,66.62,80.342039,11.148066,1044.963913,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25949,25949,0,1,3.0,0,1,0,0.0,223.99,80.342039,11.148066,971.651623,0.004566,0.067574,0.004566,0.067574,0.000000,0.000000,0.000000,0.000000
25950,25950,0,1,3.0,0,1,0,0.0,57.91,40.207765,6.022594,920.615733,0.004566,0.067574,0.004566,0.067574,0.000000,0.000000,0.000000,0.000000


In [5]:
ps_builder.accidents

,folio,timestamp,incident_level,hour,weekday,before_treatment,geometry
0,C4/160422/01377,2016-04-22 09:15:27,MIN,9,Friday,True,POINT (-99.06379 19.34598)
1,C4/160422/01752,2016-04-22 10:58:23,MIN,10,Friday,True,POINT (-99.15266 19.46846)
2,C4/160422/03130,2016-04-22 15:52:30,MIN,15,Friday,True,POINT (-99.18231 19.38936)
3,C4/160422/03245,2016-04-22 16:22:25,MIN,16,Friday,True,POINT (-99.18104 19.39155)
4,C4/160422/03568,2016-04-22 17:31:48,PIC,17,Friday,True,POINT (-99.19104 19.36336)
...,...,...,...,...,...,...,...
432702,C5/20220420/03898,2022-04-20 22:19:44,MIN,22,Wednesday,False,POINT (-99.03626 19.29632)
432703,C5/20220420/04089,2022-04-20 23:07:22,MIN,23,Wednesday,False,POINT (-99.04024 19.36653)
432704,C5/20220420/04094,2022-04-20 23:07:22,MIN,23,Wednesday,False,POINT (-99.18110 19.41348)
432705,C2C/20220420/00179,2022-04-20 21:08:35,PIC,21,Wednesday,False,POINT (-99.15056 19.44569)


In [94]:
# Propensity score matching
ps_matcher = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type="circular",
    grid_size=200,
    drop_columns=[
            "n_vialidades",
            "max_carriles",
            "max_nivel",
            "both_directions",
            "via_primaria",
            #"via_acc_cont",
            "distance_to_station"
        ]
)
ps_matcher.build()

In [96]:
effect_estimator = EffectEstimation(
    outcome=ps_matcher.outcome,
    matched_grids=ps_matcher.matched_grids,
    inicio_operaciones=ps_builder.inicio_operaciones
)
effect_estimator.estimate_all()

In [105]:
effect_estimator.get_results_table(
    outcome_type="tasas",
    variable="fcs"
).transpose().set_index('Intercept').transpose().to_clipboard(index=False)